## 针对页面中提到的 7 个步骤，可以根据实际情况替换为 OpenAI，LangChain 框架

首先，准备基础环境 和 大模型调用 函数

In [1]:
import sqlite3
import os

# 目前可以先使用 deepseek
api_key = os.environ.get("DEEPSEEK_API_KEY")
base_url = "https://api.deepseek.com"
model = "deepseek-flash"

def call_llm(prompt : str) -> str:
    print(f"[调用 LLM] Prompt: {prompt[:50]}...")
    return "对应的 LLM 返回结果"

### 1. 数据库解析 (Database Parsing)
利用 sqlite3 获取数据库中所有的表名、表结构定义（DDL）以及抽取少量真实数据样本。

In [ ]:
def parse_database(db_path: str = '../data/chinook.db') -> dict:
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # 获取所有表名
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = [row[0] for row in cursor.fetchall()]

    schema_info = {}
    for table in tables:
        # 获取表结构 (字段名、类型等)
        cursor.execute(f"PRAGMA table_info({table});")
        columns = [row[1] for row in cursor.fetchall()]

        # 抽取 2 行样本数据
        cursor.execute(f"SELECT * FROM {table} LIMIT 2;")
        samples = cursor.fetchall()

        schema_info[table] = {
            "columns": columns,
            "samples": samples
        }

    conn.close()
    return schema_info

# 测试执行
db_schema = parse_database()
db_schema

### 2. Table 选择
将用户的提问 和 基础的数据库目录发给大模型，让其挑选出需要用到的表

In [ ]:
def select_tables(question: str, db_schema: dict) -> list[str]:
    # 提取表名和简单结构作为轻量级上下文
    lightweight_schema = {table: table["columns"] for table in db_schema}

    prompt = f"""
    用户问题：{question}
    数据库中有以下表和字段：{lightweight_schema}
    请选出回答该问题需要用到的表名，以逗号分隔返回。
    """

    response = call_llm(prompt)
    # 假设 LLM 返回 "customers, invoices"
    # 这里做简单的字符串处理解析出表名列表，也可以使用 结构化输出，直接从中取
    selected_tables = [t.strip() for t in response.split(",")]
    return selected_tables

### 3. SQL 生成
将挑出的 表详细结构（含样本数据）连同问题一起交给 大模型，生成 SQL

In [ ]:
def generate_sql(question: str, selected_tables: list[str], db_schema: dict) -> str:
    # 组装被选中表的详细 Schema
    focused_schema = ""
    for table in selected_tables:
        if table in db_schema:
            focused_schema += f"表名: {table}\n"
            focused_schema += f"字段: {db_schema[table]['columns']}\n"
            focused_schema += f"样本: {db_schema[table]['samples']}\n\n"

    prompt = f"""
    用户问题：{question}
    相关表结构与数据样本如下：
    {focused_schema}
    请严格使用 SQLite 语法编写查询 SQL，只需返回 SQL 语句本身。
    """

    draft_sql = call_llm(prompt)
    return draft_sql

### 4. SQL 校验
拦截危险操作，并利用 SQLite 的 EXPLAIN QUERY PLAN 进行毫秒级的无损预检，排查“字段不存在” 或 “语法错误”

In [ ]:
import re

def validate_sql(db_path: str, sql: str) -> tuple[bool, str]:
    # 去除首尾空白字符
    clean_sql = sql.strip()
    upper_sql = clean_sql.upper()

    # 1. 白名单拦截：必须以 SELECT 或 WITH 开头
    if not (upper_sql.startswith("SELECT") or upper_sql.startswith("WITH")):
        return False, "安全拦截：系统只允许执行 SELECT 相关的查询操作！"

    # 2. 防拼接拦截：禁止多条语句 (堆叠注入)
    # 步骤 A: 允许且仅允许句末有一个分号，先将其去掉方便判断
    if clean_sql.endswith(";"):
        clean_sql = clean_sql[:-1]

    # 步骤 B: 检查 SQL 中间是否还有分号。
    # 使用正则匹配“不在单引号内的分号”。防止误杀正常的条件，如 WHERE name = 'A;B'
    if re.search(r";(?=(?:[^']*'[^']*')*[^']*$)", clean_sql):
        return False, "安全拦截：禁止使用分号拼接多条 SQL 语句！每次只能执行一条查询。"

    # 3. 预执行检查 (EXPLAIN QUERY PLAN)
    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()

        # EXPLAIN QUERY PLAN 既能检查语法和字段，又是安全的只读分析
        cursor.execute(f"EXPLAIN QUERY PLAN {clean_sql}")
        conn.close()

        return True, "校验通过"

    except sqlite3.Warning as w:
        # 兜底防御：Python 的 sqlite3.execute() 原生遇到多条语句时会抛出 Warning
        return False, f"安全拦截：检测到非法多条语句执行。报错: {str(w)}"
    except sqlite3.Error as e:
        # 返回具体的语法或字段错误给大模型去改写
        return False, f"语法或字段错误: {str(e)}"


# 测试 SQL：合法查询应通过，写错的语句和改库语句应被拦住
db_path = "../data/chinook.db"
test_sqls = [
    # 合法：客户人数
    "SELECT COUNT(*) FROM customers;",
    # 合法：各国消费汇总
    "SELECT BillingCountry, SUM(Total) FROM invoices GROUP BY BillingCountry;",
    # 字段不存在
    "SELECT CountryName FROM customers;",
    # 语法错误
    "SELECT COUNT(*) FROM customers WHERE;",
    # 删除账单
    "DELETE FROM invoices;",
    # 改金额
    "UPDATE invoices SET Total = 0 WHERE BillingCountry = 'USA';",
    # 删表
    "DROP TABLE invoices;",
    # 建表并复制邮箱 --- 应该失败
    "CREATE TABLE customer_emails AS SELECT Email FROM customers;",
]

for sql in test_sqls:
    ok, message = validate_sql(db_path, sql)
    print(ok, message, "|", sql)

### 5. SQL 改写 (SQL Rewrite)
如果校验不通过，将报错原因喂给大模型让它纠正。

In [ ]:
def rewrite_sql(question: str, bad_sql: str, error_msg: str, focused_schema: str) -> str:
    prompt = f"""
    用户问题：{question}
    表结构：{focused_schema}
    你之前生成的 SQL 报错了。
    错误的 SQL：{bad_sql}
    报错信息：{error_msg}
    请根据报错信息修正 SQL，只需返回修正后的 SQL 语句。
    """

    corrected_sql = call_llm(prompt)
    return corrected_sql

### 6. SQL 执行 (SQL Execution)
执行通过校验的正确 SQL，并捕获结果。

In [ ]:
def execute_sql(db_path: str, sql: str) -> list[tuple]:
    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        cursor.execute(sql)
        # 获取查询结果，限制条数防止 Token 溢出
        results = cursor.fetchmany(50)
        conn.close()
        return results
    except sqlite3.Error as e:
        # 实际工程中，这里的异常可以回流到“SQL 改写”节点
        print(f"执行时发生错误: {e}")
        return []

### 7. 结果汇总 (Result Synthesis)
把干巴巴的数据列表转换成自然语言汇报。

In [ ]:
def synthesize_result(question: str, sql: str, query_result: list) -> str:
    prompt = f"""
    用户问题：{question}
    执行的 SQL：{sql}
    数据库返回的结果：{query_result}
    请结合上述信息，用自然流畅、易于理解的人类语言回答用户的问题。
    如果结果为空，请如实告知。
    """

    final_answer = call_llm(prompt)
    return final_answer